# 02 — EPC: clean and prepare (Sefton, thin slice)

Requires `data/raw/epc/sefton_certificates.csv` — a Sefton-filtered export
from the "Download files" flow at
<https://get-energy-performance-data.communities.gov.uk/> (GOV.UK One Login,
filter to Sefton before downloading). Same schema as the classic EPC bulk
CSV: snake_case columns, header row present.

This notebook does three things (plan Part 2-3):
1. Keep only the fields we're actually using in Phase 1 — size, rooms, type,
   age band. Condition descriptors and the `(assumed)` filter are parked
   (plan Part 8) until the simple model is measured.
2. **Don't collapse to one row per property yet** — which certificate is
   "correct" depends on *which sale* it's being matched to (use the one
   current at that sale date, plan Part 5 leakage control #1). That
   happens in notebook 04 once the join exists.
3. Clean the address for the join in notebook 03.


In [1]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_RAW, DATA_INTERIM
import pandas as pd
import re

epc_path = DATA_RAW / "epc" / "sefton_certificates.csv"
assert epc_path.exists(), f"{epc_path} not found - see notebook 00 for how to get it."


## Load and select columns

Full EPC exports carry 90+ columns (every SAP element, cost breakdown,
etc). Phase 1 keeps only what Part 2 of the plan actually uses; everything
else stays in the raw file, parked for later.


In [2]:
KEEP_COLS = [
    "certificate_number",
    "uprn", "uprn_source",       # the join key, Route A - uprn_source: trust "Energy Assessor" over "Address Matched"
    "address1", "address2", "address3",
    "postcode",
    "property_type",
    "built_form",
    "total_floor_area",
    "number_habitable_rooms",
    "number_heated_rooms",
    "extension_count",
    "construction_age_band",
    "tenure",
    "inspection_date",
    "lodgement_date",
    "transaction_type",
]

raw = pd.read_csv(epc_path, usecols=KEEP_COLS, dtype=str, low_memory=False)
print(f"{len(raw):,} certificates loaded, {raw['uprn'].notna().mean():.1%} have a UPRN")
raw.head()


116,521 certificates loaded, 98.9% have a UPRN


,certificate_number,address1,address2,address3,postcode,built_form,construction_age_band,extension_count,inspection_date,lodgement_date,number_habitable_rooms,number_heated_rooms,property_type,tenure,total_floor_area,transaction_type,uprn,uprn_source
0,0000-2211-3070-2102-7441,24 William Morris Avenue,NaN,NaN,L20 0BQ,Mid-Terrace,England and Wales: 1950-1966,0,2022-03-09,2022-03-09,4,4,House,rented (private),68,Rental,41103209,Energy Assessor
1,0000-2419-7020-2122-6481,20 Salwick Close,NaN,NaN,PR9 9PH,Mid-Terrace,England and Wales: 1976-1982,0,2022-02-21,2022-02-21,5,5,House,owner-occupied,72,Marketed sale,41107475,Energy Assessor
2,0000-2723-6050-2109-7491,59 VALE ROAD,CROSBY,NaN,L23 5RY,Semi-Detached,England and Wales: 2003-2006,0,2021-05-07,2021-05-10,4,4,House,rented (private),78,Rental,41206797,Energy Assessor
3,0000-2854-7176-9307-3335,"9, Croxteth Avenue",NaN,NaN,L21 6NA,Mid-Terrace,England and Wales: 1900-1929,0,2013-03-04,2013-03-05,5,5,House,Owner-occupied,85,Marketed sale,41022736,Energy Assessor
4,0001-2840-7985-9377-4535,"5, Boundary Road",Litherland,NaN,L21 7LA,Semi-Detached,England and Wales: 1950-1966,0,2013-08-30,2013-09-02,5,5,House,Rented (private),84,Assessment for Green Deal,41009382,Energy Assessor


## Clean

- Floor area to numeric, drop rows without one (it's the #1 feature — no
  point carrying a certificate that can't supply it)
- Floor area sanity range: 20-500 m² (plan Part 3 cleaning table)
- Dates to datetime
- Address normalisation for the join: uppercase, strip punctuation, extract
  the leading house number and any sub-unit token (`20b`, `FLAT 4`) — plan
  Part 4, Route B


In [3]:
df = raw.copy()
df["total_floor_area"] = pd.to_numeric(df["total_floor_area"], errors="coerce")

before = len(df)
df = df.dropna(subset=["total_floor_area"])
df = df[(df["total_floor_area"] >= 20) & (df["total_floor_area"] <= 500)]
print(f"Dropped missing/out-of-range floor area: {before - len(df):,} rows")

for col in ("inspection_date", "lodgement_date"):
    df[col] = pd.to_datetime(df[col], errors="coerce")

df["postcode"] = df["postcode"].str.upper().str.strip()
df["uprn"] = pd.to_numeric(df["uprn"], errors="coerce")


Dropped missing/out-of-range floor area: 451 rows


In [4]:
def normalise_address(row) -> dict:
    """Extract house number and sub-unit for the postcode+number join
    (plan Part 4, Route B). Handles '4 Browning Road', '6, Browning Road',
    '8 BROWNING ROAD', 'Flat 4 Channel Reach', '20b Some Street'.
    """
    parts = [str(row.get(c, "")) for c in ("address1", "address2", "address3")]
    full = " ".join(p for p in parts if p and p != "nan")
    full = re.sub(r"[,]", " ", full).upper().strip()
    full = re.sub(r"\s+", " ", full)

    sub_unit = None
    m_flat = re.search(r"\b(FLAT|APARTMENT|APT|UNIT)\s*([0-9A-Z]+)\b", full)
    if m_flat:
        sub_unit = f"FLAT{m_flat.group(2)}"

    m_num = re.search(r"\b(\d+)([A-Z]?)\b", full)
    house_number = m_num.group(1) if m_num else None
    number_suffix = m_num.group(2) if m_num and m_num.group(2) else None

    return {
        "address_norm": full,
        "house_number": house_number,
        "number_suffix": number_suffix,
        "sub_unit": sub_unit,
    }

norm = df.apply(normalise_address, axis=1, result_type="expand")
df = pd.concat([df, norm], axis=1)
print(f"House number extracted for {df['house_number'].notna().mean():.1%} of rows")
df[["address1", "address_norm", "house_number", "number_suffix", "sub_unit"]].head(10)


House number extracted for 99.6% of rows


,address1,address_norm,house_number,number_suffix,sub_unit
0,24 William Morris Avenue,24 WILLIAM MORRIS AVENUE,24,NaN,NaN
1,20 Salwick Close,20 SALWICK CLOSE,20,NaN,NaN
2,59 VALE ROAD,59 VALE ROAD CROSBY,59,NaN,NaN
3,"9, Croxteth Avenue",9 CROXTETH AVENUE,9,NaN,NaN
4,"5, Boundary Road",5 BOUNDARY ROAD LITHERLAND,5,NaN,NaN
5,Roughley Bros,ROUGHLEY BROS ROSE FARM LUNT LANE,NaN,NaN,NaN
6,7 Boswell Street,7 BOSWELL STREET,7,NaN,NaN
7,101 Cambridge Road,101 CAMBRIDGE ROAD,101,NaN,NaN
8,24 Westfields Drive,24 WESTFIELDS DRIVE,24,NaN,NaN
9,2 Andrew Avenue,2 ANDREW AVENUE MELLING,2,NaN,NaN


## Multiple certificates per property

A property can have several certificates over time. Not collapsing here —
report how common repeats are, since it bounds how much the recency
question (notebook 04) will matter.


In [5]:
if df["uprn"].notna().any():
    dupe_counts = df[df["uprn"].notna()].groupby("uprn").size()
    print(f"{(dupe_counts > 1).sum():,} UPRNs have more than one certificate "
          f"({(dupe_counts > 1).mean():.1%} of UPRNs with any certificate)")
    print(dupe_counts.value_counts().sort_index().head(10))


22,403 UPRNs have more than one certificate (26.1% of UPRNs with any certificate)
1     63297
2     17555
3      3620
4       913
5       210
6        52
7        19
8        16
9         4
10        1
Name: count, dtype: int64


## Save

Renamed to match notebook 03's expected columns (`UPRN`, `POSTCODE`).


In [6]:
df = df.rename(columns={"uprn": "UPRN", "postcode": "POSTCODE"})
out = DATA_INTERIM / "epc_sefton_clean.parquet"
df.to_parquet(out, index=False)
print(f"Saved {len(df):,} rows to {out}")


Saved 116,070 rows to C:\Users\jrbah\Documents (local)\Projects\house_price_prediction\data\interim\epc_sefton_clean.parquet
